<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 2 · La pregunta antes del dato

La semana pasada viste un ranking con datos correctos y conclusión inválida: el error estaba en la
métrica, no en el cálculo. Hoy retrocedes un paso más, hasta el punto donde ese error nació. Antes de
abrir un archivo hay cuatro cosas que decidir —unidad de análisis, métrica, comparación y decisión— y
si falta una, el código que escribas después no la va a salvar. La industria de esta semana es el
**retail**: Superstore, cuatro años de pedidos de una cadena minorista estadounidense y el dataset
más famoso de la historia de la analítica de negocio. Le vas a hacer la pregunta que cualquier
gerente hace en su primer día —*«¿cuántas ventas hicimos?»*— y vas a descubrir que tiene tres
respuestas correctas y un archivo con sorpresa incluida. Hoy desbloqueas los cinco minutos de
auditoría que preceden a todo análisis y la única pregunta que separa a un analista de un generador
de reportes: ¿qué decisión cambia según el resultado?

> **Hoy haces** · Cargas Superstore y lo audita con los cuatro métodos de los cinco minutos:
> `shape`, `head`, `info` y `sample` (90 min). Descubres que el archivo trae 806 filas que no son
> ventas, y después demuestras que «una fila», «un pedido» y «un cliente» son tres cosas distintas
> que producen tres respuestas distintas a la misma pregunta. Cierras redactando la ficha de análisis
> de tu caso.
>
> **Entrega** · Este cuaderno ejecutado, los tres ejercicios resueltos y la ficha de análisis del
> proyecto de tu equipo completa en la última celda. Nombre de archivo: `lab_02_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datasets públicos de las semanas 1 a 4 viven en sitio/datos/publicos/
CANDIDATOS = [Path("../datos/publicos"), Path("datos/publicos"), Path("sitio/datos/publicos")]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    raise FileNotFoundError(
        "No encuentro la carpeta de datos. En Colab ejecuta primero:\n"
        "  !git clone https://github.com/<usuario>/CursoAnalisisDatos_IA_2026.git")

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Las cuatro preguntas que se contestan antes de abrir el archivo

Una petición de reporte suena así: *«pásame las ventas de California»*. Una pregunta de negocio suena
así: *«¿al segmento Corporate le mantenemos el descuento el próximo trimestre?»*. La primera se
contesta con una consulta; la segunda exige decidir cuatro cosas antes de escribir una línea de
código.

Si no puedes llenar las cuatro filas de la tabla siguiente, todavía no tienes un análisis: tienes
una consulta.

In [ ]:
ficha = pd.DataFrame([
    ("Unidad de análisis", "¿Qué representa una fila del resultado?",
     "Un cliente (no una línea de pedido, no un pedido)"),
    ("Métrica", "¿Qué número mide exactamente eso?",
     "Ventas acumuladas por cliente en los últimos 12 meses"),
    ("Comparación", "¿Comparado con qué?",
     "El mismo indicador en el segmento Consumer, mismo periodo"),
    ("Decisión", "¿Qué cambia según el resultado?",
     "Si Corporate no supera a Consumer, el descuento corporativo se renegocia"),
], columns=["elemento", "pregunta", "respuesta para este caso"])
ficha

📌 La comparación es la que más se olvida y la que más duele. Un número solo —«Corporate compró
706 146,37»— no es ni bueno ni malo hasta que dices contra qué. La semana 9 vuelve sobre esto con
estadística; hoy basta con no dejar la fila vacía.

## 2. Los cinco minutos: `shape`, `head`, `info` y `sample`

Superstore es la demo con la que medio mundo aprendió Tableau y Excel: pedidos de muebles, material
de oficina y tecnología en Estados Unidos entre 2015 y 2018. Precisamente porque todos creen
conocerlo, nadie lo audita. Cuatro métodos, en este orden, siempre, antes de cualquier otra cosa.

In [ ]:
ventas = pd.read_csv(DATOS / "superstore.csv")

print(f"shape : {ventas.shape[0]:,} filas × {ventas.shape[1]} columnas")
ventas.head()

`head()` impecable: pedidos con fecha, cliente, producto y monto. La tentación es empezar a calcular
ya. El segundo método existe para frenarla: `info()` dice, columna por columna, cuántos valores hay
de verdad.

In [ ]:
ventas.info()

Léelo con lápiz: el archivo tiene **10 800 filas**, pero `Order Date`, `Customer ID` y `Sales`
tienen solo **9 994 valores no nulos**. Hay 806 filas a las que les falta *todo lo que hace que una
fila sea una venta*. Y un detalle más fino: `Postal Code` tiene 9 983, es decir, 11 huecos
adicionales que sí pertenecen a ventas reales. Nada de esto se ve en `head()`, porque el principio
del archivo está limpio y ordenado: por eso existe `sample()`.

In [ ]:
ventas.sample(8, random_state=SEED)

Ahí está: entre ocho filas al azar aparece una con `Row ID` igual a **«Yes»**, un `Order ID` válido
y todo lo demás vacío. `head()` te muestra el principio del archivo, que suele estar limpio;
`sample()` te muestra el archivo de verdad. Y cuando el azar te enseña un monstruo, el siguiente
gesto es mirar dónde vive: `tail()`.

In [ ]:
print("ÚLTIMAS FILAS DEL ARCHIVO")
print(ventas.tail(4)[["Row ID", "Order ID", "Order Date", "Customer ID", "Sales"]].to_string())
print()
print("LA COSTURA: qué hay entre la fila 9 992 y la 10 001")
print(ventas.iloc[9992:10001][["Row ID", "Order ID", "Order Date", "Customer ID", "Sales"]].to_string())

El misterio queda resuelto en una pantalla. Hasta la fila 9 993 el archivo es una tabla de ventas;
justo después aparecen una fila `Person / Region`, cuatro personas con nombre y apellido, una fila
`Returned / Order ID` y ochocientas filas `Yes`. Son **las otras dos hojas del Excel original**: el
archivo nació como un libro con tres hojas —pedidos, gerentes regionales y devoluciones— y alguien
lo exportó a un solo CSV, una hoja pegada debajo de la otra. Nadie recibió un error: `read_csv` leyó
las 10 800 filas con total obediencia.

In [ ]:
basura = ventas[ventas["Sales"].isna()]

print("las 806 filas sin venta, según su primera columna:")
print(basura["Row ID"].value_counts().to_string())
print()

# La decisión mínima de hoy: análisis solo sobre líneas de pedido. La semana 4 es
# para hacer esto con protocolo; hoy basta con no sumar gerentes ni devoluciones.
lineas = ventas[ventas["Sales"].notna()].copy()
lineas["fecha"] = pd.to_datetime(lineas["Order Date"], format="%m/%d/%Y")

print(f"líneas de pedido reales : {len(lineas):,} de {len(ventas):,} filas del archivo")
print(f"periodo                 : {lineas['fecha'].min():%d-%m-%Y} a {lineas['fecha'].max():%d-%m-%Y}")
print(f"ventas totales          : {lineas['Sales'].sum():,.2f}")

In [ ]:
# ✅ Comprobación 1 · separaste las ventas de las otras dos hojas del Excel
assert len(lineas) == 9_994, \
    f"Deberían quedar 9 994 líneas de pedido y tienes {len(lineas):,}: filtra por Sales.notna(), no por otra columna"
assert abs(lineas["Sales"].sum() - 2_297_200.86) < 0.01, \
    "La suma de Sales no da 2 297 200,86: ¿borraste filas de ventas junto con la basura?"
assert lineas["Customer ID"].nunique() == 793, \
    f"Deberían ser 793 clientes y cuentas {lineas['Customer ID'].nunique()}: revisa el filtro"

print(f"Comprobación 1 superada ✓  {len(lineas):,} líneas limpias y "
      f"{lineas['Sales'].sum():,.2f} en ventas de 793 clientes")

## 3. La unidad de análisis: línea ≠ pedido ≠ cliente

Esta es la sección de la semana. El gerente pregunta *«¿cuántas ventas hicimos?»* y la respuesta
depende por completo de qué llames «una venta». El mismo archivo, tres unidades, tres números.

In [ ]:
print(f"filas del archivo (líneas de pedido) : {len(lineas):,}")
print(f"pedidos distintos (Order ID)         : {lineas['Order ID'].nunique():,}")
print(f"clientes distintos (Customer ID)     : {lineas['Customer ID'].nunique():,}")
print(f"productos distintos (Product ID)     : {lineas['Product ID'].nunique():,}")

In [ ]:
por_pedido = lineas.groupby("Order ID")["Sales"].sum()
por_cliente = lineas.groupby("Customer ID")["Sales"].sum()

niveles = pd.DataFrame({
    "unidad de análisis": ["línea de pedido", "pedido", "cliente"],
    "cuántas hay": [len(lineas), len(por_pedido), len(por_cliente)],
    "venta media": [lineas["Sales"].mean(), por_pedido.mean(), por_cliente.mean()],
    "qué pregunta contesta": [
        "¿cuánto pesa un producto dentro del carrito?",
        "¿cuánto gasta alguien cuando viene a comprar?",
        "¿cuánto vale un cliente para la empresa?",
    ],
})
niveles

📌 **229,86 · 458,61 · 2 896,85.** Los tres son «la venta promedio» y los tres son correctos. El que
el gerente quería cuando dijo «ticket promedio» es el segundo. Si le entregas el primero, la cifra
queda a la mitad; si le entregas el tercero, 6,3 veces más alta. Ninguna de las tres está mal
calculada: dos están mal elegidas.

Los factores que separan los tres niveles no son constantes, y por eso no puedes convertir uno en
otro con una regla de tres.

In [ ]:
lineas_por_pedido = lineas.groupby("Order ID").size()
pedidos_por_cliente = lineas.groupby("Customer ID")["Order ID"].nunique()

resumen = pd.DataFrame({
    "líneas por pedido": lineas_por_pedido.describe()[["mean", "50%", "min", "max"]],
    "pedidos por cliente": pedidos_por_cliente.describe()[["mean", "50%", "min", "max"]],
})
resumen.index = ["media", "mediana", "mínimo", "máximo"]
resumen

Un pedido tiene entre 1 y 14 líneas; un cliente tiene entre 1 y 17 pedidos, con una mediana de 6. Un
cliente con 17 pedidos pesa 17 veces más que uno con uno solo cuando promedias por línea, y
exactamente igual cuando promedias por cliente. Elegir la unidad de análisis **es** elegir a quién le
das voz.

## 4. Clínica de definiciones: ¿cuántos clientes tenemos?

Pregunta de directorio, aparentemente trivial. Nadie definió «tener» un cliente. Cada definición
razonable da un número distinto, y todos son defendibles.

In [ ]:
ultima = lineas["fecha"].max()
ultima_compra = lineas.groupby("Customer ID")["fecha"].max()

definiciones = [(f"compró en los últimos {d} días", int((ultima_compra >= ultima - pd.Timedelta(days=d)).sum()))
                for d in (30, 90, 180, 365)]
definiciones += [("compró alguna vez en cuatro años", int(ultima_compra.size))]

activos = pd.DataFrame(definiciones, columns=["definición de cliente activo", "cuántos son"])
activos["% de la base"] = activos["cuántos son"] / lineas["Customer ID"].nunique() * 100

fig, ax = plt.subplots(figsize=(10, 3.2))
ax.barh(activos["definición de cliente activo"], activos["cuántos son"], color="#4C72B0")
ax.invert_yaxis()
ax.set_title("La misma pregunta admite cinco respuestas: de 203 a 793 clientes activos")
ax.set_xlabel("clientes activos")
for y, valor in enumerate(activos["cuántos son"]):
    ax.text(valor + 8, y, f"{valor:,}", va="center")
plt.tight_layout()
plt.show()

activos

De 203 a 793: un factor de **3,9**. Si el directorio pide «el número de clientes activos» y tú eliges
la definición sin decirlo, estás decidiendo tú el titular de la reunión. La regla del curso: la
definición se escribe **antes** de calcular, se pone al lado del número y se mantiene igual todos los
meses. Un indicador cuya definición cambia entre trimestres no es un indicador.

## 5. La ficha de análisis

Una página, sin código, antes de tocar el teclado. Es lo que entregas a otro grupo para que la
ataque. Copia esta plantilla a una celda de texto nueva y llénala con el caso de tu equipo.

---

### Ficha de análisis · *nombre del caso*

| Campo | Contenido |
|---|---|
| **Pregunta de negocio** | *Una frase, en interrogativa, que un gerente haría en voz alta.* |
| **Unidad de análisis** | *Qué representa una fila del resultado.* |
| **Métrica** | *La fórmula exacta, con periodo y filtros.* |
| **Comparación** | *Contra qué: otro periodo, otro grupo, una meta.* |
| **Decisión que cambia** | *Qué se hace distinto según el resultado. Si no cambia nada, cambia la pregunta.* |
| **Fuente y granularidad** | *Qué archivo, qué representa una fila del origen.* |
| **Limitaciones conocidas** | *Qué no puedes contestar con estos datos.* |

---

Y la misma ficha, ejecutable, para que quede dentro del cuaderno entregado.

In [ ]:
mi_ficha = pd.Series({
    "pregunta_de_negocio": "¿Qué clientes deberían recibir la campaña de reactivación de enero?",
    "unidad_de_analisis": "Un cliente",
    "metrica": "Días desde el último pedido, medidos al 30-12-2018; ventas acumuladas de 2018",
    "comparacion": "Clientes sin pedidos en los últimos 180 días frente al resto de la base",
    "decision_que_cambia": "Quién entra en la lista de la campaña y con qué oferta",
    "fuente_y_granularidad": "superstore.csv, una fila = una línea de pedido (tras retirar 806 filas ajenas)",
    "limitaciones_conocidas": "El archivo corta el 30-12-2018: la inactividad cerca del borde no es abandono",
})
mi_ficha.to_frame("contenido")

### 🌶️ Ejercicio 1 — Guiado

Aplica los cinco minutos y los tres conteos al **segmento Consumer**: cuántas líneas, cuántos pedidos
y cuántos clientes tiene, y su venta media por pedido comparada con la global (458,61). Después
responde en una frase: ¿Consumer es «más grande» que Corporate en qué unidad, exactamente?

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: consumer = lineas[lineas["Segment"] == "Consumer"]
# Pista 2: los tres conteos son len(), ["Order ID"].nunique() y ["Customer ID"].nunique()
# Pista 3: la venta media por pedido sale de groupby("Order ID")["Sales"].sum().mean()

### 🔥 Desafío

En el archivo crudo, `ventas["Order ID"].nunique()` devuelve **5 015**. En las líneas de pedido
reales contaste **5 009**. Encuentra a los seis intrusos y explica de qué hoja del Excel vienen.
Después usa las 800 filas «Yes» para responder una pregunta que el archivo esconde: ¿cuántos pedidos
distintos fueron devueltos y qué porcentaje de los 5 009 pedidos reales representan? Termina con la
cifra corregida de pedidos que reportarías.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: set(ventas["Order ID"]) - set(lineas["Order ID"]) delata a los intrusos
# Pista 2: las filas "Yes" guardan en su columna Order ID el pedido que fue devuelto
# Pista 3: ventas.loc[ventas["Row ID"] == "Yes", "Order ID"].nunique() y divide entre 5 009

### 🎯 Reto en clase (15 min)

En equipos. Escriban la ficha de análisis de su caso —las siete filas, sin saltarse ninguna— y
entréguensela a otro equipo. El equipo receptor tiene cinco minutos para atacar dos cosas: la
**unidad de análisis** y la fila de **decisión que cambia**. Si logran demostrar que la respuesta no
cambia ninguna decisión, la ficha vuelve al remitente y se reescribe.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: copia la estructura de mi_ficha y reemplaza los siete valores por los de tu caso.
# Deja la ficha impresa en el cuaderno: es parte del entregable.

## La trampa de hoy

⚠️ **Contar filas y llamarlas ventas.** Se reconoce porque el análisis termina en un número que nadie
sabe qué cuenta. La dirección de Superstore fijó la meta del año nuevo: *«superar las 10 000
ventas»*, mirando el total de filas del archivo. Dos analistas midieron el cumplimiento el mismo
día.

In [ ]:
trampa = pd.DataFrame({
    "qué contó cada analista": ["filas del archivo (con las hojas pegadas)",
                                "líneas de pedido reales",
                                "pedidos distintos (Order ID)"],
    "'ventas' reportadas": [len(ventas), len(lineas), lineas["Order ID"].nunique()],
})
trampa["contra la meta de 10 000"] = trampa["'ventas' reportadas"].apply(
    lambda n: "cumplida ✓" if n >= 10_000 else f"faltan {10_000 - n:,}")
print(trampa.to_string(index=False))
print()
print(f"El primer analista celebra; el tercero reporta un {lineas['Order ID'].nunique() / 10_000:.1%} "
      f"de avance. Mismo archivo, mismo día: {len(ventas) / lineas['Order ID'].nunique():.1f} veces de diferencia.")

Ninguno de los dos se equivocó al programar. La diferencia es 2,2 veces y aparece **antes** de la
primera línea de código, en una palabra —«venta»— que nadie definió. Por eso la ficha de análisis va
primero: no es burocracia, es la única parte del trabajo que el asistente de IA no puede hacer por
ti, porque depende de una decisión de negocio que no está en los datos.

In [ ]:
# ✅ Comprobación 2 · las tres respuestas de la semana, verificadas
assert lineas["Order ID"].nunique() == 5_009, \
    f"Deberían ser 5 009 pedidos reales y cuentas {lineas['Order ID'].nunique():,}: ¿contaste sobre la tabla cruda (5 015)?"
assert abs(por_pedido.mean() - 458.61) < 0.01, \
    "La venta media por pedido debería ser 458,61: revisa que agrupas por Order ID antes de promediar"
assert abs(por_cliente.mean() - 2_896.85) < 0.01, \
    "La venta media por cliente debería ser 2 896,85: revisa el groupby por Customer ID"

print("Comprobación 2 superada ✓  9 994 líneas, 5 009 pedidos, 793 clientes: tres respuestas, una pregunta")

## Entregable

Sube `lab_02_apellido.ipynb` con:

- El cuaderno ejecutado de arriba a abajo, con las 806 filas ajenas identificadas y separadas.
- Los tres ejercicios resueltos. En el desafío, la cifra corregida de pedidos y el porcentaje de
  pedidos devueltos.
- Tres observaciones escritas que salgan de `info()`, `sample()` o `tail()`, con la cifra que las
  respalda.
- La ficha de análisis de tu equipo, completa, en la última celda.
- Una fila nueva en la bitácora de prompts: pídele al asistente que critique tu ficha como si fuera
  el gerente que va a recibir el resultado y anota qué objeciones aceptaste y cuáles no.

## Para tu equipo

- La empresa del proyecto se elige esta semana y no se cambia. Antes de decidir, comprueben que
  pueden conseguir un archivo real con al menos mil filas.
- La ficha de análisis es el primer entregable del proyecto integrador. La fila que más les va a
  costar es «decisión que cambia»: si no la pueden llenar, el caso no sirve, por interesante que sea.
- Pregúntense qué representa una fila del archivo de su empresa, y si el final del archivo se parece
  al principio: acaban de ver que un CSV puede traer pegadas cosas que no son de la tabla. `tail()`
  cuesta cinco segundos.